# Fine-tune Llama 3.2 1B for MathInstruct

This notebook fine-tunes `unsloth/Llama-3.2-1B-Instruct-bnb-4bit` on `TIGER-Lab/MathInstruct` with Unsloth LoRA adapters, `TrainingArguments`, and early stopping while keeping the math-tutor prompt style.

In [ ]:
%pip install torch==2.10.0 --index-url https://download.pytorch.org/whl/cu130
%pip install unsloth trl datasets huggingface_hub peft

In [ ]:
import torch
torch.__version__

In [ ]:
import os
from pathlib import Path
import unsloth
from datasets import DatasetDict, load_dataset
from huggingface_hub import login
from transformers import DataCollatorForLanguageModeling, EarlyStoppingCallback, TrainingArguments, set_seed
from trl.trainer.sft_trainer import SFTTrainer
from unsloth import FastLanguageModel

set_seed(42)

MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
DATASET_NAME = "TIGER-Lab/MathInstruct"
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR.parent if WORKDIR.name == "notebooks" else WORKDIR
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "mathinstruct-qlora-llama32-1b"
MAX_SEQ_LENGTH = 2048
MAX_EVAL_SAMPLES = 100
TRAIN_FRACTION = 0.03
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.0
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
SYSTEM_PROMPT = (
    "You are a helpful math tutor. "
    "Solve the problem with clear reasoning and end with a concise final answer."
)

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"BF16 available: {USE_BF16}")
print(f"Unsloth version: {getattr(unsloth, '__version__', 'unknown')}")
print(f"Artifacts directory: {OUTPUT_DIR}")

In [ ]:
raw_train: DatasetDict = load_dataset(DATASET_NAME, split="train")
split_dataset = raw_train.train_test_split(test_size=MAX_EVAL_SAMPLES, seed=42)

train_base = split_dataset["train"].shuffle(seed=42)
train_count = max(1, int(len(train_base) * TRAIN_FRACTION))
dataset = DatasetDict(
    {"train": train_base.select(range(train_count)),
     "val": split_dataset["test"]}
)

print(dataset)
print(f"Train fraction: {TRAIN_FRACTION} -> {len(dataset['train'])} samples")
sample = dataset["train"][0]
print("\nInstruction preview:\n")
print(sample["instruction"][:500])
print("\nReference solution preview:\n")
print(sample["output"][:500])

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=torch.bfloat16,
 )

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "up_proj",
        "down_proj",
        "o_proj",
        "gate_proj",
    ],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    random_state=42,
 )
model.config.use_cache = False
model.print_trainable_parameters()

In [ ]:
def build_messages(problem, answer=None):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem.strip()},
    ]
    if answer is not None:
        messages.append({"role": "assistant", "content": answer.strip()})
    return messages

def render_conversation(problem, answer=None, add_generation_prompt=False):
    messages = build_messages(problem, answer)
    if getattr(tokenizer, "chat_template", None):
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )

    prompt = (
        f"{SYSTEM_PROMPT}\n\n"
        f"Problem:\n{problem.strip()}\n\n"
        "Solution:\n"
    )
    if answer is None:
        return prompt
    return f"{prompt}{answer.strip()}{tokenizer.eos_token}"

def format_batch(batch):
    texts = [
        render_conversation(problem, answer)
        for problem, answer in zip(batch["instruction"], batch["output"])
    ]
    return {"text": texts}

formatted_dataset = dataset.map(
    format_batch,
    batched=True,
    remove_columns=dataset["train"].column_names,
 )

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LENGTH)

tokenized_dataset = formatted_dataset.map(
    tokenize_batch,
    batched=True,
    remove_columns=["text"],
 )

print(formatted_dataset)

In [ ]:
print(render_conversation(dataset["train"][0]["instruction"], dataset["train"][0]["output"])[:800])

In [ ]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    lr_scheduler_type="cosine",
    report_to="none",
    bf16=USE_BF16,
    fp16=torch.cuda.is_available() and not USE_BF16,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    remove_unused_columns=False,
    seed=42,
 )

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["val"],
    data_collator=data_collator,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        )
    ],
 )

train_result = trainer.train()
train_result.metrics

In [ ]:
eval_metrics = trainer.evaluate()
eval_metrics

In [ ]:
FastLanguageModel.for_inference(model)

problem = dataset["val"][0]["instruction"]
reference_solution = dataset["val"][0]["output"]
prompt = render_conversation(problem, answer=None, add_generation_prompt=True)
model_device = next(model.parameters()).device
inputs = tokenizer(prompt, return_tensors="pt").to(model_device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.1,
    )

generated_tokens = generated[0][inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("Problem:\n")
print(problem)
print("\nModel solution:\n")
print(prediction)
print("\nReference solution:\n")
print(reference_solution)

In [ ]:
final_adapter_dir = OUTPUT_DIR / "final_adapter"
final_adapter_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)

print(f"Saved LoRA adapter to: {final_adapter_dir.resolve()}")